In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
if not os.path.exists('/content/drive/MyDrive/Recipes-Recommendation-System'):
    os.makedirs("/content/drive/MyDrive/Recipes-Recommendation-System")
    %cd /content/drive/MyDrive/Recipes-Recommendation-System
    !git clone https://github.com/soheil-housni/Recipes-Recommendation-System.git \
        /content/drive/MyDrive/Recipes-Recommendation-System

else:
    %cd /content/drive/MyDrive/Recipes-Recommendation-System
    !git pull origin main

print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/Recipes-Recommendation-System
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.47 KiB | 0 bytes/s, done.
From https://github.com/soheil-housni/Recipes-Recommendation-System
 * branch            main       -> FETCH_HEAD
   c839d0a..00366a5  main       -> origin/main
Updating c839d0a..00366a5
Fast-forward
 main.ipynb                   | 232 +++++++++++++++++++------------------------
 modules/__init__.py          |   3 +-
 modules/load_parquet_file.py |   2 +-
 scaler.pkl                   | Bin 0 -> 1159 bytes
 4 files changed, 103 insertions(+), 134 deletions(-)
 create mode 100644 scaler.pkl
/content/drive/MyDrive/Recipes-Recommendation-System


In [2]:
!pip install mlflow
!pip install loguru
!pip install optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 72.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import pandas as pd
import numpy as np
import ast
import numpy as np
import torch
import mmh3
from transformers import AutoTokenizer,DistilBertModel
import itertools
import pyarrow.parquet as pq
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
import mlflow
from modules import BeforeSplitPreprocessing, AfterSplitPreprocessing, CreationDataset,CollateFunction,set_seed,seed_worker,EncodedHashedEmbeddings,Scaler,split_df,RecommendationModel,Train,BERTCreationDataset,BERTCreationDataset,BERTEmbeddingsExtractor,BertCollateFunction,ContrastiveMSELoss,OptunaFunction,load
import optuna
from loguru import logger
import gc
import os
import plotly.io as pio
import joblib
pio.renderers.default = "vscode"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
seed=42
set_seed(seed=seed)

In [5]:
mlflow.set_tracking_uri("file:./mlflow_runs/mlruns")
mlflow.set_experiment("Recipes_Recommendation_System_model_training")

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning:

The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.

2026/04/08 23:41:07 INFO mlflow.tracking.fluent: Experiment with name 'Recipes_Recommendation_System_model_training' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///content/drive/MyDrive/Recipes-Recommendation-System/mlflow_runs/mlruns/503138512899622791', creation_time=1775691667057, experiment_id='503138512899622791', last_update_time=1775691667057, lifecycle_stage='active', name='Recipes_Recommendation_System_model_training', tags={}, trace_location=None, workspace='default'>

## Creation of the processed dataframe

In [3]:
interactions_train_df=pd.read_csv("data/interactions_train.csv")
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_users_df=pd.read_csv("data/PP_users.csv")
RAW_interactions_df=pd.read_csv("data/RAW_interactions.csv")
RAW_recipes_df=pd.read_csv("data/RAW_recipes.csv")


In [17]:
interactions_train_df_filtered=interactions_train_df.sample(n=20000,replace=False,random_state=42)

In [6]:
print(interactions_train_df_filtered.shape)


(20000, 6)


In [18]:
RAW_recipes_df.rename(columns={"id":"recipe_id"},inplace=True)
PP_recipes_df.rename(columns={"techniques":"techniques_recipes"},inplace=True)
PP_users_df.rename(columns={"techniques":"techniques_users"},inplace=True)

In [8]:
print("interactions_train_df_filtered columns:")
print(interactions_train_df_filtered.columns)
print("------------------------------------------------------------------")
print("PP_recipes_df columns:")
print(PP_recipes_df.columns)
print("-------------------------------------------------------------------")
print("PP_users_df columns:")
print(PP_users_df.columns)
print("----------------------------------------------------------------------")
print("RAW_interactions_df columns:")
print(RAW_interactions_df.columns)
print("------------------------------------------------------------------------")
print("RAW_recipes_df columns:")
print(RAW_recipes_df.columns)

interactions_train_df_filtered columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'u', 'i'], dtype='object')
------------------------------------------------------------------
PP_recipes_df columns:
Index(['id', 'i', 'name_tokens', 'ingredient_tokens', 'steps_tokens',
       'techniques_recipes', 'calorie_level', 'ingredient_ids'],
      dtype='object')
-------------------------------------------------------------------
PP_users_df columns:
Index(['u', 'techniques_users', 'items', 'n_items', 'ratings', 'n_ratings'], dtype='object')
----------------------------------------------------------------------
RAW_interactions_df columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'review'], dtype='object')
------------------------------------------------------------------------
RAW_recipes_df columns:
Index(['name', 'recipe_id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dty

In [9]:
full_df=interactions_train_df_filtered.merge(PP_recipes_df,on="i",how="left",suffixes=(None,"_1")).merge(PP_users_df,on="u",how="left",suffixes=(None,"_2")).merge(RAW_interactions_df,on=["user_id","recipe_id"],how="left",suffixes=(None,"_3")).merge(RAW_recipes_df,on=["recipe_id"],how="left",suffixes=(None,"_3"))

In [10]:
full_df=full_df.drop(columns=["review",'rating_3',"id","ingredients","submitted","contributor_id","date_3","date","steps_tokens","ingredient_tokens","name_tokens","u"],axis=1)

In [11]:
print(f"Columns of the full dataframe:")
print(full_df.columns)
print("----------------------------------------")
print(f"Shape of the full dataframe:")
print(full_df.shape)
print("----------------------------------------")
print(f"Columns of the filtered interaction dataframe:")
print(interactions_train_df_filtered.shape)
print("----------------------------------------")
print(f"Head of the full  dataframe:")
display(full_df.head())
print("----------------------------------------")
for col in full_df.columns:
    print(f"Unique values of {col} :")
    print(full_df[col].unique())
    print("----------------------------------------")
for col in full_df.columns:
    print(f"The type of {col} is")
    print(full_df[col].apply(type).unique())
    print("---------------------------------")

Columns of the full dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients'],
      dtype='object')
----------------------------------------
Shape of the full dataframe:
(20000, 20)
----------------------------------------
Columns of the filtered interaction dataframe:
(20000, 6)
----------------------------------------
Head of the full  dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,ratings,n_ratings,name,minutes,tags,nutrition,n_steps,steps,description,n_ingredients
0,231160,124810,5.0,170682,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4574, 1910, 6906, 2499, 63, 332, 6270, 7470, ...","[3, 0, 0, 0, 1, 0, 0, 0, 0, 2, 1, 0, 0, 0, 0, ...","[8911, 20238, 117521, 79316, 154563, 34302, 17...",7,"[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]",7,huckleberry or blueberry coffee cake,75,"['time-to-make', 'course', 'main-ingredient', ...","[174.1, 4.0, 92.0, 8.0, 7.0, 3.0, 11.0]",11,['beat margarine and cream cheese at medium sp...,"cooking light published this in their book, fi...",10
1,142629,31342,5.0,120865,"[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",1,"[1168, 1284, 8021, 2499, 330, 6270, 3217, 2318...","[6, 0, 0, 3, 1, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, ...","[122019, 150060, 117906, 82202, 119046, 141911...",10,"[4.0, 5.0, 5.0, 5.0, 4.0, 5.0, 3.0, 5.0, 4.0, ...",10,blender quiche or whatever you have in your ...,65,"['weeknight', 'time-to-make', 'course', 'main-...","[308.4, 37.0, 8.0, 20.0, 21.0, 41.0, 3.0]",10,"['preheat oven to 350f', 'generously grease a ...",the great part is you can use whatever leftove...,12
2,822358,441841,4.0,23142,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[1648, 1706, 3347, 5298, 3044, 4623, 2148, 6270]","[1, 1, 0, 2, 3, 0, 0, 0, 1, 5, 0, 1, 0, 0, 0, ...","[29761, 87944, 72370, 139097, 39470, 37755, 23...",17,"[4.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0, 5.0, 5.0, ...",17,sweet orange coleslaw,10,"['15-minutes-or-less', 'time-to-make', 'course...","[290.1, 21.0, 71.0, 14.0, 7.0, 8.0, 13.0]",6,"['chop the apples', 'in a small bowl mix the o...",this recipe came from one of my mom's cookbook...,8
3,655579,211110,4.0,156541,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7031, 5006, 339, 3203, 5975, 6270, 590]","[10, 0, 0, 5, 11, 0, 0, 0, 0, 7, 1, 3, 0, 0, 1...","[34492, 71279, 73805, 68721, 114795, 35164, 33...",28,"[2.0, 3.0, 4.0, 2.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",28,wilted greens with garlic and balsamic vinegar,15,"['15-minutes-or-less', 'time-to-make', 'course...","[77.5, 10.0, 5.0, 5.0, 2.0, 4.0, 1.0]",5,"['cut the greens into strips 1 inch wide', 'in...",this simple but delicious side dish can be mad...,7
4,35140,114469,5.0,30839,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",2,"[1240, 6270, 5006, 1257, 2683, 5168, 800, 2461...","[176, 4, 2, 55, 72, 0, 1, 7, 4, 141, 13, 15, 0...","[121722, 36691, 151293, 154016, 1205, 12802, 1...",364,"[5.0, 5.0, 0.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",364,chicken in the limelight,75,"['time-to-make', 'course', 'main-ingredient', ...","[645.9, 69.0, 11.0, 22.0, 80.0, 54.0, 4.0]",8,"['preheat oven to 350', 'grate lime peel and s...",very tender chicken soaked with flavor,9


----------------------------------------
Unique values of user_id :
[231160 142629 822358 ... 241982 164867 137854]
----------------------------------------
Unique values of recipe_id :
[124810  31342 441841 ...  98278 274099 283283]
----------------------------------------
Unique values of rating :
[5. 4. 3. 2. 0. 1.]
----------------------------------------
Unique values of i :
[170682 120865  23142 ... 108495 128044  25867]
----------------------------------------
Unique values of techniques_recipes :
['[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [4]:
PP_users_df["len_items"]=PP_users_df["items"].str.count(r",")+1
PP_recipes_df["len_ingredients"]=PP_recipes_df["ingredient_ids"].str.count(r",")+1
max_len_ingredients=PP_recipes_df["len_ingredients"].max()
max_len_items=PP_users_df["len_items"].max()
PP_users_df.drop(columns=["len_items"],inplace=True)
PP_recipes_df.drop(columns=["len_ingredients"],inplace=True)

In [13]:
full_df_processed=BeforeSplitPreprocessing(full_df,max_len_ingredients,max_len_items)
full_df_processed=full_df_processed.preprocessing()

In [14]:
full_df_processed["rating_scaled"]=full_df_processed["rating"]/5

In [20]:
PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(int,ast.literal_eval(x))))
all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))

In [16]:
def mapping_list(x:list,mapping_serie:pd.Series):
    return [mapping_serie.loc[i] if i in mapping_serie.index else 0 for i in x]

In [17]:
full_df_processed["ingredient_ids_continuous"]=full_df_processed["ingredient_ids"].apply(lambda x : mapping_list(x,ingredient_continuous_ids))

In [18]:
full_df_processed["full_text"]=full_df_processed["name"]+" [SEP] "+full_df_processed["description"]+" [SEP] "+full_df_processed["steps"]+" [SEP] "+full_df_processed["tags"]

In [20]:
full_df_processed.to_parquet(path="./modules/dataframes/full_df_processed.parquet")


In [6]:
full_parquet_file = pq.ParquetFile("./modules/dataframes/full_df_processed.parquet")
full_df_processed=load(full_parquet_file)

In [ ]:
#df_text=full_df_processed["full_text"]

In [7]:
#distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning:


Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
#bert_dataset=BERTCreationDataset(df_text)

In [ ]:
#generator=torch.Generator()
#generator.manual_seed(seed)
#bert_collate_function_object=BertCollateFunction(tokenizer=tokenizer)
#bert_dataloader=DataLoader(dataset=bert_dataset,batch_size=32,shuffle=False,collate_fn=bert_collate_function_object.collate_fn,worker_init_fn=seed_worker,generator=generator)

In [ ]:
#device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
#bert_extractor=BERTEmbeddingsExtractor(bert_model=distilbert_model,device=device)
#bert_extractor.get_bert_embeddings(dataloader=bert_dataloader,path="./modules/BERT_embeddings")

In [8]:
all_cls_embeddings=torch.load("./modules/BERT_embeddings/all_cls_embeddings.pt")
all_mean_embeddings=torch.load("./modules/BERT_embeddings/all_mean_embeddings.pt")
print(all_cls_embeddings.shape)
print(all_mean_embeddings.shape)

torch.Size([20000, 768])
torch.Size([20000, 768])


In [9]:
train_df,val_df,test_df,train_cls_embeddings,val_cls_embeddings,test_cls_embeddings,train_mean_embeddings,val_mean_embeddings,test_mean_embeddings=split_df(full_df_processed,cls_embeddings=all_cls_embeddings,mean_embeddings=all_mean_embeddings,random_state=seed)

In [10]:
after_split_preprocessor=AfterSplitPreprocessing(train_df,val_df,test_df)
train_df_processed,val_df_processed,test_df_processed=after_split_preprocessor.preprocessing()

In [11]:
scaler=StandardScaler()
scale_processor=Scaler(scaler)
train_df_processed,val_df_processed,test_df_processed=scale_processor.scale(train_df_processed,val_df_processed,test_df_processed)

In [13]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [40]:
train_df_processed.to_parquet(path="./modules/dataframes/train_df_processed.parquet")
val_df_processed.to_parquet(path="./modules/dataframes/val_df_processed.parquet")
test_df_processed.to_parquet(path="./modules/dataframes/test_df_processed.parquet")

## Load of the dataframe

In [19]:
train_parquet_file = pq.ParquetFile("./modules/dataframes/train_df_processed.parquet")
val_parquet_file= pq.ParquetFile("./modules/dataframes/val_df_processed.parquet")
test_parquet_file= pq.ParquetFile("./modules/dataframes/test_df_processed.parquet")

train_df_processed =load(train_parquet_file)
val_df_processed=load(val_parquet_file)
test_df_processed=load(test_parquet_file)

In [14]:
print(f"Columns of the full processed dataframe:")
print(train_df_processed.columns)
print("----------------------------------------")
print(f"Shape of the full processed dataframe:")
print(train_df_processed.shape)
print("----------------------------------------")
print(f"Head of the full  processed dataframe:")
display(train_df_processed.head())
print("----------------------------------------")
for col in train_df_processed.columns:
    print(f"Exemple of {col} :")
    print(train_df_processed.iloc[0][col])
    print("----------------------------------------")
for col in train_df_processed.columns:
    print(f"The type of {col} is")
    print(train_df_processed[col].apply(type).unique())
    print("---------------------------------")

Columns of the full processed dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients',
       'rating_scaled', 'ingredient_ids_continuous', 'full_text',
       'calorie_level_scaled', 'n_items_scaled', 'n_ratings_scaled',
       'minutes_scaled', 'n_steps_scaled', 'n_ingredients_scaled'],
      dtype='object')
----------------------------------------
Shape of the full processed dataframe:
(14000, 29)
----------------------------------------
Head of the full  processed dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,...,n_ingredients,rating_scaled,ingredient_ids_continuous,full_text,calorie_level_scaled,n_items_scaled,n_ratings_scaled,minutes_scaled,n_steps_scaled,n_ingredients_scaled
3600,175727,11769,5.0,109126,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4623, 6654, 216, 6926, 6335, 4863, 0, 0, 0, 0...","[87, 2, 0, 27, 42, 1, 1, 7, 0, 80, 3, 3, 1, 0,...","[154350, 54820, 77571, 45268, 124558, 38966, 1...",202,...,6,1.0,"[4609, 6629, 215, 6900, 6313, 4847, 1, 1, 1, 1...",walnut brewery asiago cheese dip [SEP] this a ...,-1.086207,-0.386555,-0.386555,-0.156327,0.050739,-0.923525
16216,50509,350359,4.0,53777,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",2,"[5313, 5006, 3203, 5010, 7213, 588, 2594, 2856...","[127, 1, 2, 91, 191, 0, 0, 14, 3, 176, 10, 12,...","[59108, 117303, 16669, 165155, 78915, 115100, ...",580,...,11,0.8,"[5295, 4990, 3195, 4994, 7187, 586, 2588, 2849...",nif s penne with feta [SEP] a delicious and fr...,1.461012,0.004451,0.004451,-0.142037,-0.715266,0.643517
18996,9748,13098,4.0,13490,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[150, 1124, 5298, 1909, 4763, 155, 6906, 6276,...","[44, 1, 1, 23, 33, 0, 1, 5, 2, 35, 10, 5, 0, 0...","[75852, 104812, 129769, 99788, 101820, 166815,...",144,...,8,0.8,"[149, 1119, 5281, 1903, 4748, 154, 6880, 6254,...",mom s waldorf salad [SEP] this is my mother's ...,0.187403,-0.446551,-0.446551,-0.184908,-0.970601,-0.296708
14414,161282,56694,0.0,115884,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[2730, 3484, 2404, 2499, 6270, 3486, 63, 4053,...","[33, 1, 0, 10, 12, 0, 0, 5, 0, 30, 6, 3, 0, 0,...","[130597, 16122, 49703, 152649, 103416, 36541, ...",92,...,11,0.0,"[2723, 3476, 2398, 2493, 6248, 3478, 63, 4041,...",the very best salisbury steak [SEP] this is ou...,0.187403,-0.500340,-0.500340,-0.099165,-0.204596,0.643517
4882,463435,286053,4.0,60799,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[4717, 7998, 3044, 3829, 0, 0, 0, 0, 0, 0, 0, ...","[111, 1, 2, 69, 82, 0, 1, 9, 1, 114, 17, 20, 1...","[32365, 145711, 157654, 141020, 35883, 67138, ...",395,...,4,0.8,"[4702, 7969, 3036, 3820, 1, 1, 1, 1, 1, 1, 1, ...",o j yogurt shake [SEP] this is a yummy and ...,-1.086207,-0.186914,-0.186914,-0.227780,-0.970601,-1.550341


----------------------------------------
Exemple of user_id :
175727
----------------------------------------
Exemple of recipe_id :
11769
----------------------------------------
Exemple of rating :
5.0
----------------------------------------
Exemple of i :
109126
----------------------------------------
Exemple of techniques_recipes :
[1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
----------------------------------------
Exemple of calorie_level :
0
----------------------------------------
Exemple of ingredient_ids :
[4623 6654  216 6926 6335 4863    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
----------------------------------------
Exemple of techniques_users :
[87  2  0 27 42  1  1  7  0 80  3  3  1  0  3  0 40  0  0  5 20  8  2  8
  9  0  4  4 43  4  1  1  1 58  0  3 19  6 14  1  1  6 29 35  5  1 20  3
  0  5  2  3  0 18  5 18 10 20]
----------------------------------------
Exemple o

## Model

In [15]:
train_dataset=CreationDataset(train_df_processed,cls_embeddings=train_cls_embeddings,mean_embeddings=train_mean_embeddings)
val_dataset=CreationDataset(val_df_processed,cls_embeddings=val_cls_embeddings,mean_embeddings=val_mean_embeddings)

In [16]:
train_dataset.__getitem__(10)

{'user_id': np.int64(440299),
 'recipe_id': np.int64(5012),
 'rating': np.float64(5.0),
 'i': np.int64(109198),
 'techniques_recipes': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 'calorie_level': np.int64(1),
 'ingredient_ids': array([  63, 5006, 2499, 7168, 7655, 7956, 6906,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32),
 'techniques_users': array([5, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
        0, 0, 1, 0, 0, 1, 2, 0, 0, 0, 0, 2, 0, 0, 1, 0, 1, 0, 0, 1, 1, 2,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]),
 'items': array([ 18676,  75351, 109198, ...,      0,      0,      0], dtype=int32),
 'n_items': np.int64(10),
 'ratings': array([2, 5, 5, ..., 0, 0, 0], dtype=int32),
 'n_ratings': np.int64(10),
 'name': 'chicago style deep dish pizza crust',
 '

In [17]:
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunction(tokenizer=tokenizer)
collate_fn=collate_function_object.collate_fn
train_dataloader=DataLoader(train_dataset,batch_size=64,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=64,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)

In [18]:
batch=next(iter(train_dataloader))
for key in list(batch.keys()):
    print(key)
    print(f"The shape of batch {key} is :")
    print(batch[key].shape)
    print("-------------------")

user_id
The shape of batch user_id is :
torch.Size([64, 1])
-------------------
recipe_id
The shape of batch recipe_id is :
torch.Size([64, 1])
-------------------
rating_scaled
The shape of batch rating_scaled is :
torch.Size([64, 1])
-------------------
i
The shape of batch i is :
torch.Size([64, 1])
-------------------
technique_recipes
The shape of batch technique_recipes is :
torch.Size([64, 58])
-------------------
calorie_level_scaled
The shape of batch calorie_level_scaled is :
torch.Size([64, 1])
-------------------
ingredient_ids
The shape of batch ingredient_ids is :
torch.Size([64, 20])
-------------------
techniques_users
The shape of batch techniques_users is :
torch.Size([64, 58])
-------------------
items
The shape of batch items is :
torch.Size([64, 6437])
-------------------
n_items_scaled
The shape of batch n_items_scaled is :
torch.Size([64, 1])
-------------------
ratings_scaled
The shape of batch ratings_scaled is :
torch.Size([64, 6437])
-------------------
n_rat

In [ ]:
#PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(int,ast.literal_eval(x))))
#all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
#ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))

In [ ]:
#n_recipes_ids=len(np.unique(PP_recipes_df["i"]))
#n_ingredients_ids=len(all_ingredient_ids)

#print(f"There are {n_recipes_ids} recipe ids")
#print(f"There are {n_ingredients_ids} ingredient ids")

There are 178265 recipe ids
There are 7993 ingredient ids


In [ ]:
#hash_encoder=EncodedHashedEmbeddings(n_recipes_ids,512,n_ingredients_ids,512)
#hashed_recipes_ids_encoded_embeddings,hashed_ingredients_ids_encoded_embeddings=hash_encoder.get_encoded_hashed_embeddings("./modules/hashed_encoded_tables")

In [19]:
hashed_ingredients_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_ingredients.pt")
hashed_recipes_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_recipes.pt")

In [20]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print(device)

cuda


In [25]:
#Loss with alpha = 0.3 and temperature = 0.2
loss_alpha=0.3
temperature=0.2
hyperparemeters_ranges={
    "batch_size":[32, 64],
    "dropout":{
        "low":0.05,
        "high":0.3,
        "step":0.05
    },
    "projec_dropout":{
        "low":0.05,
        "high":0.3,
        "step":0.05
    },
    "lr":{
        "low":1e-4,
        "high":1e-2,
        "log":True
    },
    "weight_decay":{
        "low":1e-5,
        "high":1e-3,
        "log":True
    },
    "warmup_prop":{
        "low":0.01,
        "high":0.1,
        "step":0.01
    },
    "mean_mode": [True, False]
}
optuna_object=OptunaFunction(train_df=train_df_processed,val_df=val_df_processed,train_cls_embeddings=train_cls_embeddings,
                             val_cls_embeddings=val_cls_embeddings,train_mean_embeddings=train_mean_embeddings,
                             val_mean_embeddings=val_mean_embeddings,hashed_ingredients_ids_encoded_embeddings=hashed_ingredients_ids_encoded_embeddings,
                             hashed_recipes_ids_encoded_embeddings=hashed_recipes_ids_encoded_embeddings,loss_alpha=loss_alpha,temperature=temperature,
                             device=device,tokenizer=tokenizer,hyperparemeters_ranges=hyperparemeters_ranges)

In [30]:
study=optuna.create_study(direction="minimize",pruner=optuna.pruners.MedianPruner(),storage="sqlite:///optuna_study.db",load_if_exists=True,study_name="recommendation_model_study")

[I 2026-04-08 23:50:21,903] Using an existing study with name 'recommendation_model_study' instead of creating a new one.


In [31]:
study.optimize(optuna_object.objective,n_trials=10)

2026-04-08 23:50:40.895 | INFO     | modules.optuna_function:objective:88 - Trial number 0: 
2026-04-08 23:50:40.896 | INFO     | modules.optuna_function:objective:89 - batch_size:64_dropout:0,15000000000000002_projec_dropout:0,1_lr:0,0004724968056532259_weight_decay:1,371295256336384e-05_warmup_prop:0,060000000000000005_mean_mode:False_loss_temp:0,2_loss_alpha:0,3
2026-04-08 23:50:40.897 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-08 23:51:49.959 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 1.178026832571817
2026-04-08 23:51:49.960 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.817592550855164
2026-04-08 23:51:49.961 | INFO     | modules.train:run_training:182 - Epoch 0 : train MSE loss = 0.04678431115702752
2026-04-08 23:51:49.962 | INFO     | modules.train:run_training:184 - Epoch 0 : validation total loss = 1.163207067095715
2026-04-08 23:51:49.962 | INFO     | modules.train:run_training:185 - Epo

----------------------------------------------------------------------------------------------------


[I 2026-04-08 23:58:00,644] Trial 0 finished with value: 1.1344977643178857 and parameters: {'batch_size': 64, 'dropout': 0.15000000000000002, 'projec_dropout': 0.1, 'lr': 0.0004724968056532259, 'weight_decay': 1.371295256336384e-05, 'warmup_prop': 0.060000000000000005, 'mean_mode': False}. Best is trial 0 with value: 1.1344977643178857.
2026-04-08 23:58:01.305 | INFO     | modules.optuna_function:objective:88 - Trial number 1: 
2026-04-08 23:58:01.306 | INFO     | modules.optuna_function:objective:89 - batch_size:64_dropout:0,05_projec_dropout:0,2_lr:0,00013340658743746087_weight_decay:3,6677632134421984e-05_warmup_prop:0,08_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-08 23:58:01.307 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-08 23:59:13.465 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 1.1853736562466404
2026-04-08 23:59:13.467 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.8196088633406053


----------------------------------------------------------------------------------------------------


[I 2026-04-09 00:06:35,156] Trial 1 finished with value: 1.1371555121048638 and parameters: {'batch_size': 64, 'dropout': 0.05, 'projec_dropout': 0.2, 'lr': 0.00013340658743746087, 'weight_decay': 3.6677632134421984e-05, 'warmup_prop': 0.08, 'mean_mode': True}. Best is trial 0 with value: 1.1344977643178857.
2026-04-09 00:06:35.733 | INFO     | modules.optuna_function:objective:88 - Trial number 2: 
2026-04-09 00:06:35.734 | INFO     | modules.optuna_function:objective:89 - batch_size:32_dropout:0,1_projec_dropout:0,2_lr:0,0005119178802555773_weight_decay:1,3386144108743617e-05_warmup_prop:0,09999999999999999_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-09 00:06:35.735 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-09 00:07:46.680 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 0.9828978645992498
2026-04-09 00:07:46.681 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.171550656347035
2026-04-09 00:07:4

...

[I 2026-04-09 00:34:11,392] Trial 5 finished with value: 0.9498907532743228 and parameters: {'batch_size': 32, 'dropout': 0.05, 'projec_dropout': 0.15000000000000002, 'lr': 0.0010036326863450127, 'weight_decay': 0.00013814862105675894, 'warmup_prop': 0.08, 'mean_mode': False}. Best is trial 2 with value: 0.9452530517373033.
2026-04-09 00:34:12.007 | INFO     | modules.optuna_function:objective:88 - Trial number 6: 
2026-04-09 00:34:12.008 | INFO     | modules.optuna_function:objective:89 - batch_size:32_dropout:0,1_projec_dropout:0,1_lr:0,0017263266493212528_weight_decay:0,000683227867921707_warmup_prop:0,03_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-09 00:34:12.010 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-09 00:35:21.755 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 0.9749773392961009
2026-04-09 00:35:21.756 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.1586894907176633
2026-04-09 00:35:2

----------------------------------------------------------------------------------------------------


[I 2026-04-09 00:40:14,198] Trial 6 finished with value: 0.9534697628790333 and parameters: {'batch_size': 32, 'dropout': 0.1, 'projec_dropout': 0.1, 'lr': 0.0017263266493212528, 'weight_decay': 0.000683227867921707, 'warmup_prop': 0.03, 'mean_mode': True}. Best is trial 2 with value: 0.9452530517373033.
2026-04-09 00:40:14.801 | INFO     | modules.optuna_function:objective:88 - Trial number 7: 
2026-04-09 00:40:14.802 | INFO     | modules.optuna_function:objective:89 - batch_size:32_dropout:0,1_projec_dropout:0,1_lr:0,001532223306220127_weight_decay:0,0001182983291048804_warmup_prop:0,01_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-09 00:40:14.803 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-09 00:41:25.783 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 0.9795371995092257
2026-04-09 00:41:25.784 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.177291827016346
2026-04-09 00:41:25.785 | INFO     | mo

----------------------------------------------------------------------------------------------------


[I 2026-04-09 00:48:43,666] Trial 9 finished with value: 0.9553154892818903 and parameters: {'batch_size': 32, 'dropout': 0.1, 'projec_dropout': 0.25, 'lr': 0.006987254241678245, 'weight_decay': 1.2335097316744706e-05, 'warmup_prop': 0.09999999999999999, 'mean_mode': True}. Best is trial 2 with value: 0.9452530517373033.


In [32]:
study_loaded=optuna.load_study(study_name="recommendation_model_study",storage="sqlite:///optuna_study.db")

In [39]:
study_best_components={
    "study_best_params":study_loaded.best_params,
    "study_best_value":study_loaded.best_value
}
torch.save(study_best_components,"./train_savings/study_best_components.pt")

In [34]:
fig=optuna.visualization.plot_param_importances(study_loaded)
fig.show()

In [35]:
fig=optuna.visualization.plot_parallel_coordinate(study_loaded)
fig.show()

In [36]:
fig=optuna.visualization.plot_optimization_history(study_loaded)
fig.show()

In [5]:
best_model_name="Best_Recommendation_model"
best_model_uri=f"runs:/45f1895105314bd4a5cb348fcfd545f1/best_model_lr_0,0003897563887501594_weight_decay_6,280785308506033e-05_dropout_0,1"
result=mlflow.register_model(best_model_uri,best_model_name)

Registered model 'Best_Recommendation_model' already exists. Creating a new version of this model...
2026/04/06 21:45:30 WARNING mlflow.tracking._model_registry.fluent: Run with id 45f1895105314bd4a5cb348fcfd545f1 has no artifacts at artifact path 'best_model_lr_0,0003897563887501594_weight_decay_6,280785308506033e-05_dropout_0,1', registering model based on models:/m-8d07540e3ee34437a09f96d1c7b68df6 instead
Created version '1' of model 'Best_Recommendation_model'.


In [7]:
best_recommendation_model=mlflow.pytorch.load_model(best_model_uri,map_location=torch.device("cpu"))

2026/04/06 21:59:16 WARNING mlflow.pytorch: Stored model version '2.10.0+cu128' does not match installed PyTorch version '2.8.0+cpu'


In [11]:
for param in best_recommendation_model.parameters():
    print(param)

Parameter containing:
tensor([[ 0.0217,  0.0122,  0.0136,  ...,  0.0112,  0.0078,  0.0116],
        [-0.0037,  0.0242,  0.0092,  ..., -0.0087,  0.0165, -0.0126],
        [ 0.0278,  0.0198, -0.0101,  ..., -0.0263,  0.0063,  0.0368],
        ...,
        [ 0.0264, -0.0088, -0.0193,  ..., -0.0195,  0.0297, -0.0232],
        [ 0.0205, -0.0013, -0.0059,  ..., -0.0368,  0.0208, -0.0203],
        [-0.0152, -0.0170, -0.0115,  ...,  0.0181,  0.0044, -0.0238]],
       requires_grad=True)
Parameter containing:
tensor([ 0.0233, -0.0320, -0.0210, -0.0304, -0.0219,  0.0238,  0.0189, -0.0008,
        -0.0111,  0.0200,  0.0072,  0.0248, -0.0075, -0.0207,  0.0065,  0.0222,
        -0.0037, -0.0347, -0.0147, -0.0016,  0.0226, -0.0316, -0.0252,  0.0033,
         0.0251,  0.0166, -0.0348,  0.0034,  0.0073,  0.0114, -0.0301,  0.0092,
        -0.0051, -0.0251, -0.0001, -0.0349, -0.0116, -0.0293,  0.0181, -0.0052,
        -0.0290,  0.0200,  0.0130,  0.0205, -0.0038, -0.0281,  0.0041, -0.0332,
        -0.0170